# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to use the [`mlcroissant`](https://github.com/mlcommons/croissant) library to explore and analyze the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset.

### Dataset Source
The dataset source is provided via a Croissant schema:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

`mlcroissant` allows us to load schema metadata, enumerate record sets, list their fields, and extract/transform the data cleanly.

In [ ]:
# Ensure latest mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading

Load the dataset's Croissant metadata and display high-level dataset info.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Croissant schema URL for the dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
print(f"Name: {dataset.metadata.name}")
print(f"Version: {dataset.metadata.version}")
print(f"Description: {dataset.metadata.description}")
print(f"License: {dataset.metadata.license}")
print("\n---\n")
print("Available metadata fields:")
pprint(dataset.metadata.to_json().keys())

## 2. Data Overview

Review the available record sets, their `@id`, the corresponding fields (with `@id`s and `name`s), and what data is present in each.

--
`mlcroissant` represents tables as **record sets**. We enumerate them and, for each, enumerate their fields and columns by `@id`.

In [ ]:
# List all record sets (tables) in the dataset
print('Record sets (recordSet):')
if not dataset.metadata.recordSet:
    print('No record sets found in metadata -- loading from distribution/references...')
    # fallback: manually enumerate from distributions
    for dist in dataset.metadata.distribution or []:
        print(f"- @id: {getattr(dist, '@id', None)} name: {getattr(dist, 'name', None)}")

# If recordSet is populated, enumerate
for rset in (dataset.metadata.recordSet or []):
    print(f"- RecordSet @id: {rset['@id']}  name: {rset.get('name', 'N/A')}")
    print("  Fields:")
    fields = rset.get('field', [])
    if isinstance(fields, dict):  # rare, corner case
        fields = [fields]
    for field in fields:
        if isinstance(field, dict):
            print(f"    - Field @id: {field['@id']} name: {field.get('name', '')} col: {field.get('column', '')}")
        else:
            print(f"    - Field ref: {field}")
    print("")
# Some datasets instead provide data through distribution and can be accessed as described in Section 3

### Quick Record Preview
Let's preview the first few records for each available record set (if accessible):

In [ ]:
# Try to detect record set IDs (preferably you should copy @id as listed above).
# If recordSet is empty, the distribution may act as a single table.
# Commonly, you can use `None` as the default record set for single-table datasets.

example_recordset_id = None  # If your dataset defines recordSet, replace with its @id as found above.

try:
    sample_records = []
    for i, rec in enumerate(dataset.records(record_set=example_recordset_id)):
        if i >= 3:
            break
        sample_records.append(rec)
    print(f"Sample records from record set {example_recordset_id or '[default]'}:")
    pprint(sample_records)
except Exception as e:
    print("Could not preview records automatically.")
    print(str(e))

## 3. Data Extraction

Extract the full table(s) from the dataset, referencing all record sets by their `@id`.
We load each into a pandas DataFrame for manipulation.


In [ ]:
# If recordSet(s) are defined, enumerate their IDs; otherwise, use [None]
record_set_ids = []
if dataset.metadata.recordSet:
    for rset in dataset.metadata.recordSet:
        record_set_id = rset["@id"]
        record_set_ids.append(record_set_id)
else:
    record_set_ids.append(None)  # Single main table

dataframes = {}
for rsid in record_set_ids:
    try:
        df = pd.DataFrame(list(dataset.records(record_set=rsid)))
        dataframes[rsid or 'main'] = df
        print(f"Loaded data from record set: {rsid or '[main]'} (shape={df.shape})")
        print(f"Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Failed to load record set {rsid or '[main]'}: {e}")

# Use the first loaded table (the main one) for analysis
main_rsid = record_set_ids[0] if record_set_ids else 'main'
main_df = dataframes[main_rsid or 'main']
main_df.head()

## 4. Exploratory Data Analysis (EDA)

Let's explore and process the primary data. The most relevant numeric fields likely include those for **protein abundance**, **molecular weight**, etc.

**Note:** Pick a numeric field from the available columns above, referencing it by exact DataFrame column label (which usually matches one of the field `@id`s).

Typical tasks: filter rows by value, normalize columns, group by key fields.

In [ ]:
# Choose a numeric field from your DataFrame columns (adjust as needed!).

# Try to auto-select a likely numeric field
likely_numeric_fields = [c for c in main_df.select_dtypes(include=["number"]).columns]
if likely_numeric_fields:
    numeric_field = likely_numeric_fields[0]
else:
    # fallback: pick a field with "int", "float" or typical numeric-like name
    possible = [c for c in main_df.columns if any(x in c.lower() for x in ["mw", "weight", "abundance", "count", "number", "mass", "peptide"])]
    if possible:
        numeric_field = possible[0]
    else:
        # Last-resort: just pick the first column
        numeric_field = main_df.columns[0]

print(f"Analyzing numeric field: {numeric_field}")

# Drop NAs for this field, coerce to float if possible
main_df[numeric_field] = pd.to_numeric(main_df[numeric_field], errors="coerce")
analysis_df = main_df.dropna(subset=[numeric_field]).copy()

# Filter: e.g. values greater than 10
threshold = 10
filtered_df = analysis_df[analysis_df[numeric_field] > threshold]
print(f"Filtered records where '{numeric_field}' > {threshold}: {filtered_df.shape[0]}")
print(filtered_df[[numeric_field]].head())

# Normalize
filtered_df[numeric_field + "_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / (filtered_df[numeric_field].std() or 1)

print(f"Top normalized entries for {numeric_field}:")
print(filtered_df[[numeric_field, numeric_field + "_normalized"]].head())

# Try grouping by another field: e.g. 'description', or first string-type field
group_candidate = None
for col in main_df.columns:
    if col != numeric_field and main_df[col].dtype == object and main_df[col].nunique() < 15:
        group_candidate = col
        break

if group_candidate:
    print(f"\nGrouping by: {group_candidate}")
    grouped = filtered_df.groupby(group_candidate)[numeric_field].mean().sort_values(ascending=False)
    print(grouped.head())

## 5. Visualization

Let's visualize the distribution of the main numeric field, and relationships if possible.

In [ ]:
import matplotlib.pyplot as plt

# Histogram of the numeric field
plt.figure(figsize=(7,4))
filtered_df[numeric_field].hist(bins=30)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If there is a possible grouping field, show mean by group
if group_candidate:
    plt.figure(figsize=(10,4))
    group_means = filtered_df.groupby(group_candidate)[numeric_field].mean().sort_values(ascending=False)
    group_means.plot(kind="bar")
    plt.title(f"Mean {numeric_field} by {group_candidate}")
    plt.ylabel(f"Mean {numeric_field}")
    plt.xlabel(group_candidate)
    plt.show()

## 6. Conclusion

This notebook demonstrated how to use `mlcroissant` to load, explore, and analyze the FAIR² dataset *Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells* from a Croissant schema. We enumerated available tables (record sets), loaded records by their `@id`, processed and filtered numeric fields, normalized and grouped data, and visualized key results. For further work, customize the analysis with your biological research questions or downstream ML tasks.